In [ ]:
## import libraries
## Date: 8 Nov, 2023

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## check the files
import os
path = '/kaggle/input/data-science-london-scikit-learn'
# x = os.listdir(path)[0]
# pd.read_csv(path + '/'+ x)

os.listdir(path)

In [ ]:
## read the data
X   = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/train.csv', header=None)
y = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/trainLabels.csv', header=None)
y = np.ravel(y)
test_data    = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/test.csv', header=None)

In [ ]:
## know the available styles in plt
print(plt.style.available)
## use a style
plt.style.use('ggplot')

In [ ]:
## KNN for classification
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score

## split the data
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
N = range(2, 26)
kfold = 10
test_acc = []
val_acc = []
best_acc = 0.0
best_knn = None

for n in N:
    knn = KNeighborsClassifier(n_neighbors=n)
    knn.fit(x_train, y_train)
    test_acc.append(knn.score(x_train, y_train))
    val_acc.append(np.mean(cross_val_score(knn, X, y, cv=kfold)))
    
    ## get best acc, knn
    if np.mean(cross_val_score(knn, X, y, cv=kfold)) > best_acc:
        best_acc = np.mean(cross_val_score(knn, X, y, cv=kfold))
        best_knn = knn


In [ ]:
## plot the accuracy
# plt.figure(figsize=(25, 25))
plt.plot(test_acc, c = 'b', label = 'test_acc')
plt.plot(val_acc, c = 'r', label = 'val_acc')
plt.xlabel('Number of Neighbors')
plt.ylabel('Accuracy')
plt.title('K Neighbors vs Accuracy')
plt.legend()
plt.show()

## print best_acc, best_knn
print('best_acc:', best_acc)
print('best_knn:', best_knn)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5).fit(x_train, y_train)
knn.score(x_test, y_test)

In [ ]:
## save the output
y_pred = knn.predict(test_data)
submission = pd.DataFrame()
submission['Id'] = range(1, len(test_data) + 1)
submission['Solution'] = y_pred
submission.to_csv('/kaggle/working/submission.csv', index=False)
submission

In [ ]:
## Scaling the data
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer

X_st   = StandardScaler().fit_transform(X)
X_min  = MinMaxScaler().fit_transform(X)
X_norm = Normalizer().fit_transform(X)

In [ ]:
K_neigh = range(2, 26)
best_k = 1
kfold = 10
best_acc = 0.0
best_knn = None
best_scaling = None
val_acc = {'std': [], 'min': [], 'norm': []}

for k in K_neigh:
    knn = KNeighborsClassifier(n_neighbors=k)
    
    s1  = np.mean(cross_val_score(knn, X_st, y, cv=kfold))
    val_acc['std'].append(s1)
    
    s2  = np.mean(cross_val_score(knn, X_min, y, cv=kfold))
    val_acc['min'].append(s2)
    
    s3  = np.mean(cross_val_score(knn, X_norm, y, cv=kfold))
    val_acc['norm'].append(s3)
    
    ## check best_acc
    if s1 > best_acc:
        best_acc = s1
        best_knn = knn
        best_k = k
        best_scaling = 'std'
        
    elif s2 > best_acc:
        best_acc = s2
        best_knn = knn
        best_k = k
        best_scaling = 'min'
        
    elif s3 > best_acc:
        best_acc = s3
        best_knn = knn
        best_k = k
        best_scaling = 'norm'
    

In [ ]:
## Plot the accuracy
plt.figure(figsize=(10, 5))
plt.plot(val_acc['std'], c='r', label='std')
plt.plot(val_acc['min'], c='g', label='min')
plt.plot(val_acc['norm'], c='b', label='norm')
plt.title('K Neighbors vs Accuracy')
plt.xlabel('Number of Neighbors')
plt.ylabel('Accuracy')
plt.xticks(K_neigh)
plt.legend()
plt.show()

In [ ]:
## print best
print('best_acc:', best_acc)
print('best knn:', best_knn)
print('best_k:', best_k)
print('best_scaling:', best_scaling)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X_norm, y, test_size=0.2, random_state=42)
knn = KNeighborsClassifier(n_neighbors=best_k).fit(x_train, y_train)
knn.score(x_test, y_test)

In [ ]:
## save the output
y_pred = knn.predict(test_data)
submission = pd.DataFrame()
submission['Id'] = range(1, len(test_data) + 1)
submission['Solution'] = y_pred
submission.to_csv('/kaggle/working/submission_norm.csv', index=False)
submission

In [ ]:
## Draw the correlation
plt.figure(figsize=(20, 20))
sns.heatmap(pd.DataFrame(X_norm).corr(), annot=True, linewidths=0.5, fmt='0.1f')

In [ ]:
## Use RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV

kfold = 10
best_rf = None
best_acc = 0.0
cv_range = range(5, 11)
val_acc = []
n_features = []

from tqdm import tqdm

for cv in tqdm(cv_range):
    rf_clf = RandomForestClassifier()
    rfecv = RFECV(rf_clf, cv=cv, step=1, scoring='accuracy').fit(X_norm, y)
    
    val_acc.append(np.mean(cross_val_score(rf_clf, X_norm[:, rfecv.support_], y, cv=kfold)))
    n_features.append(rfecv.n_features_)
    
    if val_acc[-1] > best_acc:
        best_acc = val_acc[-1]

In [ ]:
## Draw
plt.plot(cv_range, val_acc, label='CV Accuracy')
for i in range(len(cv_range)):
    plt.annotate(str(n_features[i]), xy=(cv_range[i], val_acc[i]))
plt.legend()
plt.xlabel('Kfold')
plt.ylabel('Accuracy')
plt.title('CV vs Accuracy')
plt.show()

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=100)
rfecv = RFECV(rf_clf, cv=10, step=1, scoring='accuracy').fit(X_norm, y)

In [ ]:
X_new = X_norm[: , rfecv.support_]
x_train, x_test, y_train, y_test = train_test_split(X_new, y, test_size=0.3, random_state=42)
rf_clf.fit(x_train, y_train)
rf_clf.score(x_test, y_test)

In [ ]:
test_data = np.array(test_data)
submission = pd.DataFrame()
submission['Id'] = range(1, len(test_data) +1)
submission['Solution'] = rf_clf.predict(test_data[:, rfecv.support_])

submission.to_csv('/kaggle/working/submission_feature.csv', index=False)

In [ ]:
## Search Grid
from sklearn.model_selection import GridSearchCV

## the models
knn = KNeighborsClassifier()
rf  = RandomForestClassifier()

## the params
param_grid = {'n_neighbors': range(2, 10), 'weights': ['uniform', 'distance']}

## GridSearchCV for knn
grid_knn = GridSearchCV(knn, param_grid, cv=10, scoring='accuracy').fit(X, y)
print(f'best estimator: {grid_knn.best_estimator_}\nbest score: {grid_knn.best_score_}')

In [ ]:
## GridSearchCV for RandomForestClassifier
param_grid = {'n_estimators': range(10, 100, 10), 'criterion': ['gini', 'entropy']}
grid_rf    = GridSearchCV(rf, param_grid, verbose=3, cv=10 ,scoring='accuracy').fit(X, y)
print(f'best estimator: {grid_rf.best_estimator_}\nbest score: {grid_rf.best_score_}')

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=90).fit(x_train, y_train)
rf.score(x_test, y_test)

In [ ]:
import numpy as np
#import sklearn as sk
#import matplotlib.pyplot as plt
import pandas as pd

#from sklearn.linear_model import LogisticRegression
#from sklearn.linear_model import Perceptron
#from sklearn import tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.ensemble import VotingClassifier
#from sklearn import svm

#### READING OUR GIVEN DATA INTO PANDAS DATAFRAME ####
x_train = np.asarray(X)
y_train =y
x_test = np.asarray(test_data)
print('training_x Shape:',x_train.shape,',training_y Shape:',y_train.shape, ',testing_x Shape:',x_test.shape)

#Checking the models
x_all = np.r_[x_train,x_test]
print('x_all shape :',x_all.shape)

#### USING THE GAUSSIAN MIXTURE MODEL ####
from sklearn.mixture import GaussianMixture
lowest_bic = np.infty
bic = []
n_components_range = range(1, 7)
cv_types = ['spherical', 'tied', 'diag', 'full']
for cv_type in cv_types:
    for n_components in n_components_range:
        # Fit a mixture of Gaussians with EM
        gmm = GaussianMixture(n_components=n_components,covariance_type=cv_type)
        gmm.fit(x_all)
        bic.append(gmm.aic(x_all))
        if bic[-1] < lowest_bic:
            lowest_bic = bic[-1]
            best_gmm = gmm
            
best_gmm.fit(x_all)
x_train = best_gmm.predict_proba(x_train)
x_test = best_gmm.predict_proba(x_test)


#### TAKING ONLY TWO MODELS FOR KEEPING IT SIMPLE ####
knn = KNeighborsClassifier()
rf = RandomForestClassifier()

param_grid = dict( )
#### GRID SEARCH for BEST TUNING PARAMETERS FOR KNN #####
grid_search_knn = GridSearchCV(knn,param_grid=param_grid,cv=10,scoring='accuracy').fit(x_train,y_train)
print('best estimator KNN:',grid_search_knn.best_estimator_,'Best Score', grid_search_knn.best_estimator_.score(x_train,y_train))
knn_best = grid_search_knn.best_estimator_

#### GRID SEARCH for BEST TUNING PARAMETERS FOR RandomForest #####
grid_search_rf = GridSearchCV(rf, param_grid=dict( ), verbose=3,scoring='accuracy',cv=10).fit(x_train,y_train)
print('best estimator RandomForest:',grid_search_rf.best_estimator_,'Best Score', grid_search_rf.best_estimator_.score(x_train,y_train))
rf_best = grid_search_rf.best_estimator_


knn_best.fit(x_train,y_train)
print(knn_best.predict(x_test)[0:10])
rf_best.fit(x_train,y_train)
print(rf_best.predict(x_test)[0:10])

#### SCORING THE MODELS ####
print('Score for KNN :',cross_val_score(knn_best,x_train,y_train,cv=10,scoring='accuracy').mean())
print('Score for Random Forest :',cross_val_score(rf_best,x_train,y_train,cv=10,scoring='accuracy').max())

### IN CASE WE WERE USING MORE THAN ONE CLASSIFIERS THEN VOTING CLASSIFIER CAN BE USEFUL ###
#clf = VotingClassifier(
#		estimators=[('knn_best',knn_best),('rf_best',rf_best)],
#		#weights=[871856020222,0.907895269918]
#	)
#clf.fit(x_train,y_train)
#print clf.predict(x_test)[0:10]

##### FRAMING OUR SOLUTION #####
knn_best_pred = pd.DataFrame(knn_best.predict(x_test))
rf_best_pred = pd.DataFrame(rf_best.predict(x_test))
#voting_clf_pred = pd.DataFrame(clf.predict(x_test))

knn_best_pred.index += 1
rf_best_pred.index += 1
#voting_clf_pred.index += 1

rf_best_pred.columns = ['Solution']
rf_best_pred['Id'] = np.arange(1,rf_best_pred.shape[0]+1)
rf_best_pred = rf_best_pred[['Id', 'Solution']]
print(rf_best_pred)

#knn_best_pred.to_csv('knn_best_pred.csv')
rf_best_pred.to_csv('Submission_rf.csv', index=False)
#voting_clf_pred.to_csv('voting_clf_pred.csv')